# Assignment 5

Perform logistic regression to classify if a patient has a benign tumor or malignant tumor (cancer) based on the features provided. Generate the confusion matrix and evaluate the quality of the model by computing relevant performance metrics including Precision, Recall, accuracy, F1-Score etc. Plot the ROC curve and calculate AUC.
Dataset: samples_cancer.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve, roc_auc_score
)

# 1. Load Real Dataset and Clean BareNuc Non-Numeric Values
df = pd.read_csv('samples_cancer.csv')
df = df[pd.to_numeric(df['BareNuc'], errors='coerce').notnull()]
df['BareNuc'] = df['BareNuc'].astype(int)
X, y = df.drop(columns=['ID', 'Class']), df['Class'].map({2: 0, 4: 1})

# 2. Train-Test Split (75/25) and Feature Scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 3. Fit Logistic Regression Model
clf = LogisticRegression(solver='liblinear', random_state=42)
clf.fit(X_train, y_train)
y_pred, y_prob = clf.predict(X_test), clf.predict_proba(X_test)[:, 1]

# 4. Performance Metrics
cm = confusion_matrix(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

print("=== Real Data: Logistic Regression Cancer Performance ===")
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
print(f"F1-Score: {f1:.4f} | AUC: {auc:.4f}")
print(f"Confusion Matrix:\n{cm}")

# 5. Plot Confusion Matrix & ROC Curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
im = ax1.imshow(cm, cmap='Blues')
ax1.set_title('Confusion Matrix (Benign vs Malignant)')
ax1.set_xlabel('Predicted Label'); ax1.set_ylabel('True Label')
for i in range(2):
    for j in range(2):
        ax1.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                 color="white" if cm[i, j] > cm.max()/2 else "black")

ax2.plot(fpr, tpr, color='darkorange', lw=2.5,
         label=f'ROC Curve (AUC = {auc:.4f})')
ax2.plot([0, 1], [0, 1], color='navy', linestyle='--', lw=1.5)
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('Receiver Operating Characteristic (ROC)')
ax2.legend(loc="lower right"); ax2.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout(); plt.savefig('logistic_regression_roc.png', dpi=300)
plt.show()